# Regex Basics

# Basic Matching Functions


Use Regex for pattern matching in texts
- to check if a pattern exist -  re\.search('pattern', text)
- to get all instances of a particular pattern - re.findall('pattern', text)
- to split strings - re.split('pattern', text)

In [2]:
import re

In [3]:
text = "This is a good day."

# if there is a match, the re.search returns a match project
# (this is considered to pass the "if" check)
if re.search("good", text):
    print('Matched')

# if there is no match, the re.search returns None
print(re.search("come", text) == None)

Matched
True


In [4]:
# findall() finds all instances
text = "Amy works diligently. Amy gets good grades. Our student Amy is succesful."
print(re.findall("Amy", text))

# split separates the string (the keyword is excluded)
print(re.split('Amy', text))

['Amy', 'Amy', 'Amy']
['', ' works diligently. ', ' gets good grades. Our student ', ' is succesful.']


In [5]:
# to match pattern at the start or end of a string, use anchors (^ for start, $ for end)
text = "Amy works diligently. Amy gets good grades. Our student Amy is succesful."
print(re.search("^Amy",text))
print(re.search("succesful.$", text))
print(re.search("Jack$", text))

<re.Match object; span=(0, 3), match='Amy'>
<re.Match object; span=(63, 73), match='succesful.'>
None


# Patterns and Character Classes
Some commonly used special characters
- . (dot) matches any character except a new line
- ^ (caret) matches the start of the string
- $ matches the end of string or just before the new line
- \* match 0 or more repetitions of the preceding RE
- \+ match 1 or more repetitions of the preceding RE
- [] the set operator
- \w matches Unicode word characters and the underscore (_)
- \W matches the opposite of /w
- \d matches any Unicode decimal digit including [0-9] and other digit characters
- \D matches the opposite of /d
- \s matches Unicode whitespace. This includes [ \t\n\r\f\v] and others.

In [6]:
grades="ACAAAABCBCBAAD"

print(re.findall("B",grades)) # Find all instances of 'B'
print(re.findall("AB",grades)) # Find instances of A followed by B

['B', 'B', 'B']
['AB']


In [7]:
# The square brackets are the set operator []
grades="ACAAAABCBCBAAD"
print(re.findall("[AB]",grades)) # find instances of A's or B's
print(re.findall("[A][B-D]",grades)) # find instances of A followed by a B or a C or a D
print(re.findall("AB|AC",grades)) # find instances of AB or AC, without []
print(re.findall("^A",grades)) # find instances of A at the start of the string
print(re.findall("[^A]",grades)) # find instances of not A
print(re.findall("^[^A]",grades)) # find instances of not A, at the start of the string

['A', 'A', 'A', 'A', 'A', 'B', 'B', 'B', 'A', 'A']
['AC', 'AB', 'AD']
['AC', 'AB']
['A']
['C', 'B', 'C', 'B', 'C', 'B', 'D']
[]


# Quantifiers

In [8]:
grades="ACAAAABCBCBAAD"
print(re.findall("A{2,10}",grades)) # find A that appears consecutively 2 to 10 times
print(re.findall("A{1,1}A{1,1}",grades)) # find an A followed by another A
print(re.findall("A{2, 2}",grades)) # Note: there should be no space in {} quantifier
print(re.findall("A{2}",grades)) # equivalent to {2,2}


['AAAA', 'AA']
['AA', 'AA', 'AA']
[]
['AA', 'AA', 'AA']


In [9]:
# Example, find any decreasing trend in the student's grades
grades="ACAAAABCBCBAAD"
re.findall("A{1,10}B{0,10}C{0,10}D{0,10}",grades)

['AC', 'AAAABC', 'AAD']

#### Example with wiki data
In this example, we first import the data. We find that headers all have the words '[edit]' behind them followed by a new line character.

In [19]:
with open("data_intro/ferpa.txt","r") as file:
    wiki=file.read()
print(wiki[0:300])

Overview[edit]
FERPA gives parents access to their child's education records, an opportunity to seek to have the records amended, and some control over the disclosure of information from the records. With several exceptions, schools must have a student's consent prior to the disclosure of education 


In [20]:
# find all instances of headers
# By placing an r directly before the opening quote,
# you tell Python to treat all backslashes as literal text rather than escape characters.
re.findall(r"[a-zA-Z]{1,100}\[edit\]",wiki)

['Overview[edit]', 'records[edit]', 'records[edit]']

In [21]:
for title in re.findall(r"[\w ]*\[edit\]",wiki):
    print(re.split(r"[\[]",title)[0])

Overview
Access to public records
Student medical records


# Groups
- Used to match multiple patterns in one shot and return tuples

In [31]:
# The first group is ([\w ]*), the second group is (\[edit\])
re.findall(r"([\w ]*)(\[edit\])",wiki)

[('Overview', '[edit]'),
 ('Access to public records', '[edit]'),
 ('Student medical records', '[edit]')]

In [37]:
# Use re.finditer() to print out each item in the group
# the .group(n) method return item number 1 in each group
for x in re.finditer(r"([\w ]*)(\[edit\])",wiki):
    print(x.group(1))

Overview
Access to public records
Student medical records


In [42]:
# Naming groups, use (?P<name>) at start of each group
# First group name: ?P<title>
# Second group name: P<edit_link>
for item in re.finditer(r"(?P<title>[\w ]*)(?P<edit_link>\[edit\])",wiki):
    # We can get the dictionary returned for the item with .groupdict()
    print(item.groupdict())
print("--------------------------------------------------------")
for item in re.finditer(r"(?P<title>[\w ]*)(?P<edit_link>\[edit\])",wiki):
    print(item.groupdict()['title'])

{'title': 'Overview', 'edit_link': '[edit]'}
{'title': 'Access to public records', 'edit_link': '[edit]'}
{'title': 'Student medical records', 'edit_link': '[edit]'}
--------------------------------------------------------
Overview
Access to public records
Student medical records


In [44]:
# Look-ahead matching - matching what comes before the [edit] pattern
# after matching, through the [edit] away.
# use (?=) before the pattern
for item in re.finditer(r"(?P<title>[\w ]+)(?=\[edit\])",wiki):
    # What this regex says is match two groups,
    # the first will be named and called title, will have any amount
    # of whitespace or regular word characters,
    # the second will be the characters [edit] but we don't actually
    # want this edit put in our output match objects
    print(item)

<re.Match object; span=(0, 8), match='Overview'>
<re.Match object; span=(2715, 2739), match='Access to public records'>
<re.Match object; span=(3692, 3715), match='Student medical records'>


#### Example - buddhist.txt data

In [56]:
with open("data_intro/buddhist.txt","r", encoding='utf8') as file:
    wiki=file.read()
print(wiki[200:1000])

prove this article by adding citations to reliable sources. Unsourced material may be challenged and removed.
Find sources: "Buddhist universities and colleges in the United States" – news · newspapers · books · scholar · JSTOR (December 2009) (Learn how and when to remove this template message)
There are several Buddhist universities in the United States. Some of these have existed for decades and are accredited. Others are relatively new and are either in the process of being accredited or else have no formal accreditation. The list includes:

Dhammakaya Open University – located in Azusa, California, part of the Thai Wat Phra Dhammakaya[1]
Dharmakirti College – located in Tucson, Arizona Now called Awam Tibetan Buddhist Institute (http://awaminstitute.org/)
Dharma Realm Buddhist Univers


In [66]:
# the pattern: university followed by - then 'located in' city and state
# to match long patterns, we can use the verbose mode with """
pattern = r"""
(?P<title>.*)       # the university title (named)
(–\ located\ in\ )  # note that endash, not hyphen, is used.
(?P<city>\w*)       # city the university is in (named)
(,\ )               # separator for the state
(?P<state>\w*)      # the state the city is located in
"""

for item in re.finditer(pattern, wiki, re.VERBOSE):
    print(item.groupdict())

{'title': 'Dhammakaya Open University ', 'city': 'Azusa', 'state': 'California'}
{'title': 'Dharmakirti College ', 'city': 'Tucson', 'state': 'Arizona'}
{'title': 'Dharma Realm Buddhist University ', 'city': 'Ukiah', 'state': 'California'}
{'title': 'Ewam Buddhist Institute ', 'city': 'Arlee', 'state': 'Montana'}
{'title': 'Institute of Buddhist Studies ', 'city': 'Berkeley', 'state': 'California'}
{'title': 'Maitripa College ', 'city': 'Portland', 'state': 'Oregon'}
{'title': 'University of the West ', 'city': 'Rosemead', 'state': 'California'}
{'title': 'Won Institute of Graduate Studies ', 'city': 'Glenside', 'state': 'Pennsylvania'}


#### Example - NY times and Hashtags

In [71]:
with open("data_intro/nytimeshealth.txt","r", encoding="utf-8") as file:
    # We'll read everything into a variable and take a look at it
    health=file.read()
print(health[:700])

548662191340421120|Sat Dec 27 02:10:34 +0000 2014|Risks in Using Social Media to Spot Signs of Mental Distress http://nyti.ms/1rqi9I1
548579831169163265|Fri Dec 26 20:43:18 +0000 2014|RT @paula_span: The most effective nationwide diabetes prevention program you've probably never heard of:  http://newoldage.blogs.nytimes.com/2014/12/26/diabetes-prevention-that-works/
548579045269852161|Fri Dec 26 20:40:11 +0000 2014|The New Old Age Blog: Diabetes Prevention That Works http://nyti.ms/1xm7fTi
548444679529041920|Fri Dec 26 11:46:15 +0000 2014|Well: Comfort Casseroles for Winter Dinners http://nyti.ms/1xTNoO0
548311901227474944|Fri Dec 26 02:58:39 +0000 2014|High-Level Knowledge Before Veterans A


In [76]:
# we want to find any pattern of a hashtag followed by some texts
pattern = r'#[\w\d]*(?=\s)'
print(re.findall(pattern, health))

['#askwell', '#pregnancy', '#Colorado', '#VegetarianThanksgiving', '#FallPrevention', '#Ebola', '#Ebola', '#ebola', '#Ebola', '#Ebola', '#EbolaHysteria', '#AskNYT', '#Ebola', '#Ebola', '#Liberia', '#Excalibur', '#ebola', '#Ebola', '#dallas', '#nobelprize2014', '#ebola', '#ebola', '#monrovia', '#ebola', '#nobelprize2014', '#ebola', '#nobelprize2014', '#Medicine', '#Ebola', '#Monrovia', '#Ebola', '#smell', '#Ebola', '#Ebola', '#Ebola', '#Monrovia', '#Ebola', '#ebola', '#monrovia', '#liberia', '#benzos', '#ClimateChange', '#Whole', '#Wheat', '#Focaccia', '#Tomatoes', '#Olives', '#Recipes', '#Health', '#Ebola', '#Monrovia', '#Liberia', '#Ebola', '#Ebola', '#Liberia', '#Ebola', '#blood', '#Ebola', '#organtrafficking', '#EbolaOutbreak', '#SierraLeone', '#Freetown', '#SierraLeone', '#ebolaoutbreak', '#kenema', '#ebola', '#Ebola', '#ebola', '#ebola', '#Ebola', '#ASMR', '#AIDS2014', '#AIDS', '#MH17', '#benzos']
